# PDF Extractor — Enhanced for Hydrology AI
**Features:**
- Enhanced `clean_text` → removes headers/footers, fixes hyphenated line-breaks, strips citations
- Image extraction → saves to disk **+** base64 in records (with caption context)
- Metadata inference from filename/path
- Document splitting into LangChain `Document` chunks
- Global `all_docs` list for Qdrant/Chroma ingestion
- Folder-level and single-file chunk builders

**Install:**
```bash
pip install pdfplumber pymupdf langchain langchain-text-splitters
```

---
## Cell 1 — Imports & Global State

In [1]:
import re
import json
import base64
import sys
from io import BytesIO
from pathlib import Path
from typing import List, Dict, Any, Optional

import pdfplumber          # text + table extraction
import fitz                # PyMuPDF — image extraction
from PIL import Image      # image save/convert  (pip install Pillow)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Global document store ─────────────────────────────────────────────────────
# All chunked LangChain Documents land here — ready for Qdrant / Chroma upload
all_docs: List[Document] = []

print("Imports OK")

Imports OK


---
## Cell 2 — Enhanced `clean_text`
Handles the specific noise found in academic PDFs (journal headers, page numbers, hyphenated line-breaks, inline citations).

In [2]:
# ── Patterns compiled once at module level for speed ─────────────────────────

# Matches journal header/footer lines like:
#   "Environmental Science and Pollution Research (2023) 30:59327–59348"
#   "59328  Environmental Science and Pollution Research ..."
# Also catches generic "Vol.:(0123456789) 1 3" Springer artefacts
_RE_JOURNAL_HEADER = re.compile(
    r"^\s*"
    r"(?:"
    r"\d{4,6}\s+[A-Z][^\n]{10,}\(\d{4}\)[^\n]{0,60}"  # page-num + journal name
    r"|[A-Z][^\n]{10,}\(\d{4}\)\s+\d+:\d+[–\-]\d+"      # journal name + volume
    r"|Vol\.:\(\d+\)[^\n]*"                               # Vol.:(0123456789)
    r"|1\s+3\s*$"                                         # trailing "1 3" springer mark
    r")",
    re.MULTILINE,
)

# Matches inline citations like (Tehrany et al. 2019), (Al-Juaidi 2020),
# (Smith and Jones 2018), (Wu et al. 2019, 2020)
_RE_CITATION = re.compile(
    r"\("
    r"(?:[A-Z][a-záéíóú'\-]+(?:\s+et\s+al\.?|\s+and\s+[A-Z][a-z]+)?)"
    r"(?:\s+\d{4}(?:[,;]\s*\d{4})*)"
    r"\)",
)

# Hyphenated line-break:  "flood-\ncausing"  →  "flood-causing"
#                         "sus-\nceptibility" → "susceptibility"
_RE_HYPHEN_BREAK = re.compile(r"(\w+)-\n(\w+)")

# Soft line-break inside a paragraph (single newline between text lines)
_RE_SOFT_BREAK = re.compile(r"(?<!\n)\n(?!\n)")

# CID garbage tokens: (cid:123)
_RE_CID = re.compile(r"\(cid:\d+\)")

# Figure / Table caption lines:  "Fig. 3 Altitude and slope map"
#                                 "Table 2 The Jarque–Bera ..."
_RE_CAPTION = re.compile(
    r"^\s*(?:Fig\.?|Figure|Table|Eq\.|Equation)\s+\d+[a-z]?[^\n]{0,120}$",
    re.MULTILINE | re.IGNORECASE,
)

# Springer page-marker lines: "© Springer", "\x00 Springer"
_RE_SPRINGER = re.compile(r"^\s*[©\x00\u2019]?\s*Springer[^\n]*$", re.MULTILINE)

# Excessive whitespace
_RE_MULTI_BLANK = re.compile(r"\n{3,}")


def clean_text(
    text: str,
    remove_citations: bool = True,
    remove_captions: bool = False,   # set True if you don't want fig/table captions
) -> str:
    """
    Deep-clean raw PDF text for vector embedding.

    Steps applied in order:
      1. Replace non-breaking spaces and CID garbage
      2. Strip journal header / footer lines
      3. Strip Springer watermark lines
      4. Fix hyphenated line-breaks  (flood-\ncausing → flood-causing)
      5. Collapse soft line-breaks inside paragraphs
      6. Optionally remove inline citations  (Al-Juaidi et al. 2020)
      7. Optionally strip figure/table caption lines
      8. Collapse multiple blank lines → one blank line
      9. Strip leading/trailing whitespace per line and globally
    """
    if not text:
        return ""

    # 1 — normalise unicode whitespace & CID tokens
    text = text.replace("\u00a0", " ")
    text = _RE_CID.sub("", text)

    # 2 — drop journal header / footer lines
    text = _RE_JOURNAL_HEADER.sub("", text)

    # 3 — drop Springer watermark lines
    text = _RE_SPRINGER.sub("", text)

    # 4 — fix hyphenated line-breaks
    #     "flood-\ncausing" → "flood-causing"
    #     "sus-\nceptibility" → "susceptibility"  (removes the hyphen entirely)
    def _fix_hyphen(m: re.Match) -> str:
        word_a, word_b = m.group(1), m.group(2)
        # If the combined form without hyphen looks like a real word,
        # drop the hyphen; otherwise keep it (e.g. "flood-causing" keeps hyphen)
        combined = word_a + word_b
        hyphenated = word_a + "-" + word_b
        # Simple heuristic: if word_a ends with a consonant cluster → it was split
        # across lines without a real hyphen → merge without hyphen
        if len(word_a) >= 3 and word_a[-1] not in "aeiouAEIOU-":
            return combined
        return hyphenated

    text = _RE_HYPHEN_BREAK.sub(_fix_hyphen, text)

    # 5 — collapse soft line-breaks (single \n inside paragraphs) → space
    text = _RE_SOFT_BREAK.sub(" ", text)

    # 6 — remove inline citations
    if remove_citations:
        text = _RE_CITATION.sub("", text)

    # 7 — strip figure/table caption lines
    if remove_captions:
        text = _RE_CAPTION.sub("", text)

    # 8 — collapse 3+ consecutive blank lines → one blank line
    text = _RE_MULTI_BLANK.sub("\n\n", text)

    # 9 — strip trailing spaces per line & global strip
    lines = [line.rstrip() for line in text.split("\n")]
    return "\n".join(lines).strip()


# # ── Quick smoke-test ──────────────────────────────────────────────────────────
# if __name__ == "__main__" or True:
#     _sample = (
#         "Environmental Science and Pollution Research (2023) 30:59327–59348\n"
#         "1 3\n"
#         "Flash floods arise when rainfall exceeds the take-up rate of the\n"
#         "drainage system basins (Tehrany et al. 2019). Topographic slope is con-\n"
#         "sidered an important feature that determines the harshness of\n"
#         "flooding.\n"
#         "© Springer\n"
#         "Fig. 3 Altitude and slope map\n"
#     )
#     print("--- BEFORE ---")
#     print(_sample)
#     print("--- AFTER  ---")
#     print(clean_text(_sample))

---
## Cell 3 — Table Extraction Helper

In [3]:
def extract_tables_from_page(page) -> List[Dict[str, Any]]:
    """
    Extract all tables on a pdfplumber page.
    Returns list of dicts with:
      - rows      : 2-D list of cell strings
      - markdown  : pipe-delimited string ready for embedding
    """
    tables_found = []
    for table in page.extract_tables():
        if not table:
            continue
        cleaned_rows = [
            [str(cell).strip() if cell is not None else "" for cell in row]
            for row in table
        ]
        md_lines = []
        for i, row in enumerate(cleaned_rows):
            md_lines.append(" | ".join(row))
            if i == 0:
                md_lines.append(" | ".join(["---"] * len(row)))
        tables_found.append({"rows": cleaned_rows, "markdown": "\n".join(md_lines)})
    return tables_found

---
## Cell 4 — Image Extraction (save to disk + base64)
Uses **PyMuPDF (fitz)** which gives pixel-level access to embedded images.  
Each image record carries:
- `image_path` — where the PNG was saved on disk  
- `base64` — for multimodal embedding / direct API calls  
- `caption_context` — 300 chars of surrounding page text (for vector DB)

In [4]:
def extract_images_from_pdf(
    pdf_path: str | Path,
    output_dir: str | Path,
    min_width: int = 100,
    min_height: int = 100,
) -> List[Dict[str, Any]]:
    """
    Extract all embedded images from a PDF using PyMuPDF.

    Parameters
    ----------
    pdf_path   : path to the source PDF
    output_dir : folder where PNG files will be saved
    min_width  : skip tiny images (icons, bullets) smaller than this
    min_height : skip tiny images smaller than this

    Returns
    -------
    List of image records, each dict contains:
        page         (int)  — 1-indexed page number
        image_index  (int)  — position on page
        image_path   (str)  — absolute path to saved PNG
        width        (int)
        height       (int)
        base64       (str)  — base64-encoded PNG bytes
        caption_context (str) — surrounding text snippet (for vector DB)
    """
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_records: List[Dict[str, Any]] = []
    doc = fitz.open(str(pdf_path))

    for page_num, page in enumerate(doc, start=1):
        # Get surrounding text for caption context
        page_text = page.get_text("text")
        page_text_clean = clean_text(page_text)          # reuse our cleaner
        caption_snippet = page_text_clean[:300].replace("\n", " ").strip()

        image_list = page.get_images(full=True)
        for img_idx, img_info in enumerate(image_list):
            xref = img_info[0]  # xref is the image object reference

            try:
                base_image = doc.extract_image(xref)
            except Exception as e:
                print(f"  [WARN] Could not extract image xref={xref} on page {page_num}: {e}")
                continue

            img_bytes = base_image["image"]
            img_ext   = base_image.get("ext", "png")
            width     = base_image.get("width", 0)
            height    = base_image.get("height", 0)

            # Skip tiny images (bullets, logos, artifacts)
            if width < min_width or height < min_height:
                continue

            # Always save as PNG for consistency
            stem = pdf_path.stem
            img_filename = f"{stem}_page{page_num:03d}_img{img_idx:02d}.png"
            img_save_path = output_dir / img_filename

            try:
                # Convert to PIL then save as PNG (handles JPEG/JBIG2 sources)
                pil_img = Image.open(BytesIO(img_bytes))
                pil_img.save(str(img_save_path), format="PNG")
            except Exception as e:
                print(f"  [WARN] PIL save failed for page {page_num} img {img_idx}: {e}")
                # Fallback: write raw bytes
                img_save_path = output_dir / f"{stem}_page{page_num:03d}_img{img_idx:02d}.{img_ext}"
                img_save_path.write_bytes(img_bytes)

            # Encode to base64 (always from the saved PNG for consistency)
            b64_str = base64.b64encode(img_save_path.read_bytes()).decode("utf-8")

            image_records.append({
                "page":            page_num,
                "image_index":     img_idx,
                "image_path":      str(img_save_path.resolve()),
                "width":           width,
                "height":          height,
                "base64":          b64_str,
                "caption_context": caption_snippet,
            })

    doc.close()
    print(f"[IMAGES] Extracted {len(image_records)} images → {output_dir}")
    return image_records

---
## Cell 5 — Per-page & Full-PDF Extraction

In [5]:
def extract_page(page, page_num: int) -> Dict[str, Any]:
    """
    Extract cleaned text + tables from a single pdfplumber page.
    x/y_tolerance=3 groups characters within 3pt gaps.
    """
    raw_text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""
    text     = clean_text(raw_text)        # ← enhanced cleaner
    tables   = extract_tables_from_page(page)
    return {
        "page":       page_num,
        "text":       text,
        "tables":     tables,
        "has_tables": bool(tables),
    }


def extract_pdf(
    pdf_path: str | Path,
    image_output_dir: Optional[str | Path] = None,
    extract_images: bool = True,
) -> Dict[str, Any]:
    """
    Process an entire PDF: text, tables, and (optionally) images.

    Parameters
    ----------
    pdf_path         : path to PDF
    image_output_dir : where to save extracted images;
                       defaults to  <pdf_dir>/images/<pdf_stem>/
    extract_images   : set False to skip image extraction entirely

    Returns
    -------
    dict with keys:
        source, file_name, total_pages,
        pages  (list of per-page dicts),
        images (list of image records),
        full_text (str, for RAG/embedding)
    """
    pdf_path = Path(pdf_path)

    # ── TEXT + TABLE extraction via pdfplumber ────────────────────────────
    pages_data = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        total = len(pdf.pages)
        print(f"[INFO] '{pdf_path.name}' — {total} pages")
        for i, page in enumerate(pdf.pages, start=1):
            print(f"  → Extracting text page {i}/{total}...", end="\r")
            pages_data.append(extract_page(page, page_num=i))
    print()

    # ── IMAGE extraction via PyMuPDF ──────────────────────────────────────
    image_records: List[Dict[str, Any]] = []
    if extract_images:
        if image_output_dir is None:
            image_output_dir = pdf_path.parent / "images" / pdf_path.stem
        image_records = extract_images_from_pdf(
            pdf_path=pdf_path,
            output_dir=image_output_dir,
        )

    # ── Build full_text for RAG ────────────────────────────────────────────
    parts = []
    for p in pages_data:
        if p["text"]:
            parts.append(f"[PAGE {p['page']}]\n{p['text']}")
        for t in p["tables"]:
            parts.append(f"[TABLE — PAGE {p['page']}]\n{t['markdown']}")

    return {
        "source":      str(pdf_path),
        "file_name":   pdf_path.name,
        "total_pages": total,
        "pages":       pages_data,
        "images":      image_records,
        "full_text":   "\n\n".join(parts),
    }

---
## Cell 6 — Flatten for Embedding
Converts the extracted doc into flat records (text chunks + image records) ready for a vector DB.

In [6]:
def flatten_for_embedding(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Convert extracted doc into flat records for vector DB / splitter.

    Record kinds:
      'text'  — page text block
      'table' — markdown table (embeds well as prose)
      'image' — image with base64 + caption context

    Each record carries full metadata so you can filter by page,
    kind, source in Qdrant/Chroma.
    """
    records: List[Dict[str, Any]] = []

    base = {
        "source":    doc["source"],
        "file_name": doc["file_name"],
    }

    # TEXT + TABLE records
    for page in doc["pages"]:
        page_meta = {**base, "page": page["page"]}

        if page["text"]:
            records.append({**page_meta, "kind": "text", "text": page["text"]})

        for idx, table in enumerate(page["tables"], start=1):
            records.append({
                **page_meta,
                "kind":        "table",
                "table_index": idx,
                "text":        table["markdown"],  # markdown embeds better
                "rows":        table["rows"],
            })

    # IMAGE records  (base64 + caption context stored for multimodal use)
    for img in doc.get("images", []):
        records.append({
            **base,
            "kind":            "image",
            "page":            img["page"],
            "image_index":     img["image_index"],
            "image_path":      img["image_path"],
            "width":           img["width"],
            "height":          img["height"],
            "base64":          img["base64"],
            # caption_context embeds into vector DB so you can ask
            # "Explain this flood map" and retrieve the right image
            "text":            f"[IMAGE page {img['page']}] {img['caption_context']}",
        })

    return records

---
## Cell 7 — Metadata Inference
Extended from your original markdown version — now also handles PDF paths (`source_type = 'pdf'`).

In [ ]:
APP_DATA_ROOT = Path(r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\app_data")


def is_app_data_path(file_path: str | Path) -> bool:
    """Return True only for files under app_data directory."""
    try:
        path = Path(file_path).resolve()
        return APP_DATA_ROOT.resolve() in path.parents or path == APP_DATA_ROOT.resolve()
    except Exception:
        return "app_data" in str(file_path).lower().replace("\\", "/")


def infer_doc_metadata(file_path: str | Path) -> dict:
    """
    Infer metadata from path but keep only app_data documents.
    Non-app_data documents are explicitly marked as excluded.
    """
    path = Path(file_path)
    file_name = path.name.lower()
    folder_name = path.parent.name.lower()
    suffix = path.suffix.lower()          # .md or .pdf

    source_type = "pdf" if suffix == ".pdf" else "markdown"
    in_app_data = is_app_data_path(path)

    metadata = {
        "source": path.name,
        "file_name": path.name,
        "source_type": source_type,
        "dataset": "app_data" if in_app_data else "excluded",
        "doc_type": "general",
        "app": "hydrology_app",
        "module": "app_knowledge" if in_app_data else "excluded",
        "topic": "application" if in_app_data else "excluded",
        "section": "app_data" if in_app_data else folder_name,
        "is_app_data": in_app_data,
    }

    # Strict guard: everything outside app_data is excluded from retrieval/indexing.
    if not in_app_data:
        return metadata

    # Optional doc-type rules for app documents
    if "walkthrough" in file_name:
        metadata["doc_type"] = "walkthrough"
    elif "report" in file_name:
        metadata["doc_type"] = "report"
    elif "architecture" in file_name:
        metadata["doc_type"] = "architecture"
    elif "workflow" in file_name:
        metadata["doc_type"] = "workflow"
    elif "setup" in file_name:
        metadata["doc_type"] = "setup"
    elif "readme" in file_name:
        metadata["doc_type"] = "readme"
    elif "analysis" in file_name:
        metadata["doc_type"] = "analysis"
   

    return metadata

---
## Cell 8 — `split_into_documents`
Your chunking function — unchanged, inserted as-is.

In [8]:
def split_into_documents(
    base_doc: Document,
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Split a LangChain Document into smaller chunks.
    Separators are ordered: section headers → paragraphs → lines → words.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_text(base_doc.page_content)
    chunk_docs = []
    for i, chunk in enumerate(chunks):
        new_metadata = {
            **base_doc.metadata,
            "chunk_id":   i,
            "chunk_size": len(chunk),
        }
        chunk_docs.append(Document(page_content=chunk, metadata=new_metadata))
    return chunk_docs

---
## Cell 9 — PDF → `all_docs` Pipeline
Convert a PDF directly into chunked LangChain Documents and append to `all_docs`.

In [ ]:
def build_chunk_documents_from_pdf(
    pdf_path: str | Path,
    image_output_dir: Optional[str | Path] = None,
    chunk_size: int = 300,
    chunk_overlap: int = 50,
    extract_images: bool = True,
    save_json: bool = False,
) -> List[Document]:
    """
    Full pipeline: PDF -> extract -> clean -> chunk -> append to global all_docs.
    Only files inside APP_DATA_ROOT are accepted.
    """
    global all_docs
    pdf_path = Path(pdf_path)

    file_meta = infer_doc_metadata(pdf_path)
    if not file_meta.get("is_app_data", False):
        print(f"[SKIP] Non app_data file ignored: {pdf_path}")
        return all_docs

    # 1. Extract PDF
    doc = extract_pdf(
        pdf_path=pdf_path,
        image_output_dir=image_output_dir,
        extract_images=extract_images,
    )

    # 2. Flatten to embedding records
    flat_records = flatten_for_embedding(doc)

    # 3. Convert each record to a LangChain Document & chunk
    for record in flat_records:
        text = record.get("text", "").strip()
        if not text:
            continue

        record_meta = {
            **file_meta,
            "page": record.get("page"),
            "kind": record.get("kind"),
        }

        if record["kind"] == "image":
            record_meta["image_path"] = record.get("image_path")
            record_meta["image_index"] = record.get("image_index")
            record_meta["base64"] = record.get("base64")
            record_meta["width"] = record.get("width")
            record_meta["height"] = record.get("height")
            all_docs.append(Document(page_content=text, metadata=record_meta))

        elif record["kind"] == "table":
            record_meta["table_index"] = record.get("table_index")
            all_docs.append(Document(page_content=text, metadata=record_meta))

        else:
            base_doc = Document(page_content=text, metadata=record_meta)
            chunks = split_into_documents(base_doc, chunk_size, chunk_overlap)
            all_docs.extend(chunks)

    # 4. Optionally save flat records to JSON
    if save_json:
        json_records = [{k: v for k, v in r.items() if k != "base64"} for r in flat_records]
        out = pdf_path.parent / f"{pdf_path.stem}.json"
        out.parent.mkdir(parents=True, exist_ok=True)
        with open(out, "w", encoding="utf-8") as f:
            json.dump(json_records, f, ensure_ascii=False, indent=2)
        print(f"[SAVED JSON] {out}")

    print(f"[all_docs] Total app_data docs after PDF: {len(all_docs)}")
    return all_docs

---
## Cell 10 — Markdown Loaders (unchanged from your originals)

In [10]:
def load_markdown_file(
    file_path: str | Path,
    custom_metadata: Optional[dict] = None,
) -> List[Document]:
    """
    Load a single markdown file into a LangChain Document with inferred metadata.
    """
    path = Path(file_path)
    text = path.read_text(encoding="utf-8")
    metadata = infer_doc_metadata(path)
    if custom_metadata:
        metadata.update(custom_metadata)
    return [Document(page_content=text, metadata=metadata)]


def load_markdown_folder(
    folder_path: str | Path,
    app_name: str = "hydrology_app",
) -> List[Document]:
    """
    Recursively load all .md files from a folder.
    """
    folder_path = Path(folder_path)
    docs = []
    for md_file in sorted(folder_path.rglob("*.md")):
        metadata = infer_doc_metadata(md_file)
        metadata["app"] = app_name
        text = md_file.read_text(encoding="utf-8")
        docs.append(Document(page_content=text, metadata=metadata))
    print(f"[MARKDOWN] Loaded {len(docs)} files from {folder_path}")
    return docs

---
## Cell 11 — Chunk Builders (Folder & Single File)
Your exact implementations — inserted as-is.

In [11]:
def build_chunk_documents(
    folder_path: str,
    app_name: str = "hydrology_app",
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Build chunked documents from ALL markdown files in a folder.
    Appends results to global all_docs.
    """
    global all_docs
    base_docs = load_markdown_folder(folder_path=folder_path, app_name=app_name)
    for base_doc in base_docs:
        chunk_docs = split_into_documents(
            base_doc=base_doc,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        all_docs.extend(chunk_docs)
    return all_docs


def build_chunk_documents_files(
    file_path: str,
    app_name: str = "hydrology_app",
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Build chunked documents from ONE markdown file.
    Appends results to global all_docs.
    """
    global all_docs
    base_docs = load_markdown_file(
        file_path=file_path,
        custom_metadata={"app": app_name},
    )
    for base_doc in base_docs:
        chunk_docs = split_into_documents(
            base_doc=base_doc,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        all_docs.extend(chunk_docs)
    return all_docs

---
## Cell 12 — Usage Examples
Adjust paths to match your local setup.

In [ ]:
# =============================================================================
# EXAMPLE A - Process a single app_data PDF
# =============================================================================

from pathlib import Path

APP_DATA_ROOT = Path(r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\app_data")
IMAGE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\images\app_data_images"

pdf_candidates = sorted(APP_DATA_ROOT.rglob("*.pdf"))
if not pdf_candidates:
    print(f"[ERROR] No PDFs found in app_data path: {APP_DATA_ROOT}")
else:
    PDF_PATH = str(pdf_candidates[0])
    print(f"[INFO] Using PDF: {PDF_PATH}")

    all_docs = []   # reset before each run if you want a clean slate

    build_chunk_documents_from_pdf(
        pdf_path=PDF_PATH,
        image_output_dir=IMAGE_DIR,
        chunk_size=300,
        chunk_overlap=50,
        extract_images=True,
        save_json=True,
    )

    print(f"\nTotal docs in all_docs : {len(all_docs)}")
    text_chunks = [d for d in all_docs if d.metadata.get("kind") == "text"]
    table_chunks = [d for d in all_docs if d.metadata.get("kind") == "table"]
    image_chunks = [d for d in all_docs if d.metadata.get("kind") == "image"]
    print(f"  text  chunks : {len(text_chunks)}")
    print(f"  table chunks : {len(table_chunks)}")
    print(f"  image chunks : {len(image_chunks)}")

    if all_docs:
        first = all_docs[0]
        print("\nSample metadata :", {k: v for k, v in first.metadata.items() if k != "base64"})
        print("Sample content  :", first.page_content[:300])

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXAMPLE B — Build chunks from a markdown FOLDER (appends to all_docs)
# ══════════════════════════════════════════════════════════════════════════════

DOCS_FOLDER = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\current_process"

build_chunk_documents(
    folder_path=DOCS_FOLDER,
    app_name="hydrology_app",
    chunk_size=300,
    chunk_overlap=50,
)

print(f"Base folder          : {DOCS_FOLDER}")
print(f"Total chunks in all_docs: {len(all_docs)}")
if all_docs:
    print("Sample chunk metadata:", all_docs[0].metadata)
    print("Sample chunk preview :", all_docs[0].page_content[:200])

[MARKDOWN] Loaded 3 files from C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\current_process
Base folder          : C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\current_process
Total chunks in all_docs: 125
Sample chunk metadata: {'source': 'Feature Extraction for Mahanadi Basin Flood Susceptibility for my project.md', 'file_name': 'Feature Extraction for Mahanadi Basin Flood Susceptibility for my project.md', 'source_type': 'markdown', 'doc_type': 'general', 'app': 'hydrology_app', 'module': 'flood_susceptibility', 'topic': 'flood', 'section': 'current_process', 'chunk_id': 0, 'chunk_size': 108}
Sample chunk preview : # **Feature Extraction Workflow for Flood Susceptibility Mapping**

# **Hydrology Project — Mahanadi Basin**


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXAMPLE C — Build chunks from a SINGLE markdown file (appends to all_docs)
# ══════════════════════════════════════════════════════════════════════════════

FILE_PATH = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\flood_data\generated_knowledge\flood_knowledge.md"

build_chunk_documents_files(
    file_path=FILE_PATH,
    app_name="hydrology_app",
    chunk_size=300,
    chunk_overlap=50,
)

print(f"Base file            : {FILE_PATH}")
print(f"Total chunks in all_docs: {len(all_docs)}")
if all_docs:
    print("Sample chunk metadata:", all_docs[0].metadata)
    print("Sample chunk preview :", all_docs[0].page_content[:200])

Base file            : C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\flood_data\generated_knowledge\flood_knowledge.md
Total chunks in all_docs: 325
Sample chunk metadata: {'source': 'flood_knowledge.md', 'file_name': 'flood_knowledge.md', 'source_type': 'markdown', 'doc_type': 'general', 'app': 'hydrology_app', 'module': 'flood_susceptibility', 'topic': 'flood', 'section': 'generated_knowledge', 'chunk_id': 0, 'chunk_size': 37}
Sample chunk preview : # Flood Susceptibility Knowledge Base


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
#  EXAMPLE D — Inspect a specific image chunk (verify base64 roundtrip)
# ══════════════════════════════════════════════════════════════════════════════

from IPython.display import Image as IPImage, display

img_docs = [d for d in all_docs if d.metadata.get("kind") == "image"]

if img_docs:
    sample_img = img_docs[0]
    print("Image metadata (no base64):")
    print({k: v for k, v in sample_img.metadata.items() if k != "base64"})
    print("\nCaption context:", sample_img.page_content)

    # Decode base64 and display inline in the notebook
    b64 = sample_img.metadata.get("base64", "")
    if b64:
        img_bytes = base64.b64decode(b64)
        display(IPImage(data=img_bytes))
else:
    print("No image docs in all_docs yet — run Example A first.")

No image docs in all_docs yet — run Example A first.


---
## Cell 13 — `build_chunk_documents_from_pdf_folder`

Walks a folder **recursively**, processes every `.pdf` found, and appends all chunks to `all_docs`.

**Behaviour:**
- Recurses into all subfolders (`rglob('*.pdf')`)
- Skips corrupted / password-protected PDFs with a warning — never stops the run
- All extracted images land in **one flat image folder** (configurable)
- Prints a per-PDF summary line and a final report
- Optionally saves a `.json` sidecar per PDF (base64 stripped)

In [ ]:
def build_chunk_documents_from_pdf_folder(
    folder_path: str | Path,
    image_output_dir: Optional[str | Path] = None,
    chunk_size: int = 300,
    chunk_overlap: int = 50,
    extract_images: bool = True,
    save_json: bool = False,
) -> List[Document]:
    """
    Recursively process all app_data PDFs only and append chunks to global all_docs.
    """
    global all_docs

    folder_path = Path(folder_path)
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    if not is_app_data_path(folder_path):
        raise ValueError(f"Only app_data folder is allowed. Got: {folder_path}")

    if image_output_dir is None:
        image_output_dir = folder_path / "images"
    image_output_dir = Path(image_output_dir)
    image_output_dir.mkdir(parents=True, exist_ok=True)

    all_pdfs = sorted(folder_path.rglob("*.pdf"))
    if not all_pdfs:
        print(f"[WARN] No PDF files found under: {folder_path}")
        return all_docs

    print(f"[FOLDER] Found {len(all_pdfs)} app_data PDF(s) under: {folder_path}")
    print(f"[FOLDER] Images -> {image_output_dir}")
    print("-" * 60)

    run_stats = {
        "total_pdfs": len(all_pdfs),
        "succeeded": 0,
        "failed": 0,
        "failed_files": [],
        "total_text": 0,
        "total_tables": 0,
        "total_images": 0,
        "docs_before": len(all_docs),
    }

    for idx, pdf_path in enumerate(all_pdfs, start=1):
        print(f"\n[{idx}/{len(all_pdfs)}] Processing: {pdf_path.name}")
        docs_before = len(all_docs)

        try:
            build_chunk_documents_from_pdf(
                pdf_path=pdf_path,
                image_output_dir=image_output_dir,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                extract_images=extract_images,
                save_json=save_json,
            )

            new_docs = all_docs[docs_before:]
            n_text = sum(1 for d in new_docs if d.metadata.get("kind") == "text")
            n_tables = sum(1 for d in new_docs if d.metadata.get("kind") == "table")
            n_images = sum(1 for d in new_docs if d.metadata.get("kind") == "image")

            run_stats["succeeded"] += 1
            run_stats["total_text"] += n_text
            run_stats["total_tables"] += n_tables
            run_stats["total_images"] += n_images

            print(
                f"  +{len(new_docs)} docs "
                f"(text={n_text}, table={n_tables}, image={n_images}) "
                f"-> all_docs total: {len(all_docs)}"
            )

        except Exception as exc:
            run_stats["failed"] += 1
            run_stats["failed_files"].append(str(pdf_path))
            print(f"  SKIPPED - {type(exc).__name__}: {exc}")
            continue

    docs_added = len(all_docs) - run_stats["docs_before"]
    print("\n" + "=" * 60)
    print("APP_DATA FOLDER PROCESSING COMPLETE")
    print("=" * 60)
    print(f"  PDFs found      : {run_stats['total_pdfs']}")
    print(f"  Succeeded       : {run_stats['succeeded']}")
    print(f"  Failed/skipped  : {run_stats['failed']}")
    if run_stats["failed_files"]:
        for f in run_stats["failed_files"]:
            print(f"    - {f}")
    print(f"  Docs added      : {docs_added}")
    print(f"    text chunks   : {run_stats['total_text']}")
    print(f"    table chunks  : {run_stats['total_tables']}")
    print(f"    image docs    : {run_stats['total_images']}")
    print(f"  all_docs total  : {len(all_docs)}")
    print("=" * 60)

    return all_docs

---
## Cell 14 — `all_docs` Inspector
Handy helper — call any time to see a breakdown of what's currently in `all_docs`.

In [16]:
def inspect_all_docs(sample_n: int = 2) -> None:
    """
    Print a structured summary of the global all_docs list.
    Shows counts by kind, by source file, and prints sample chunks.
    """
    global all_docs

    if not all_docs:
        print("all_docs is empty.")
        return

    from collections import Counter

    kind_counts   = Counter(d.metadata.get("kind", "unknown") for d in all_docs)
    source_counts = Counter(d.metadata.get("file_name", "unknown") for d in all_docs)
    module_counts = Counter(d.metadata.get("module", "unknown") for d in all_docs)

    print(f"{'='*55}")
    print(f"  all_docs  —  {len(all_docs)} total documents")
    print(f"{'='*55}")

    print("\n── By kind ──────────────────────────────────")
    for kind, count in kind_counts.most_common():
        bar = "█" * min(count, 40)
        print(f"  {kind:<10} {count:>5}  {bar}")

    print("\n── By source file ───────────────────────────")
    for src, count in source_counts.most_common():
        print(f"  {count:>5}  {src}")

    print("\n── By module (inferred metadata) ────────────")
    for mod, count in module_counts.most_common():
        print(f"  {count:>5}  {mod}")

    print(f"\n── Sample chunks (first {sample_n}) ──────────────────")
    for i, doc in enumerate(all_docs[:sample_n]):
        safe_meta = {k: v for k, v in doc.metadata.items() if k != "base64"}
        print(f"\n  [{i}] metadata : {safe_meta}")
        print(f"       content  : {doc.page_content[:200]!r}")

    print(f"{'='*55}")


# Quick call — comment out if you don't want it to run on import
# inspect_all_docs()

In [23]:
all_docs: List[Document] = []
len(all_docs)

0

---
## Cell 15 — Usage: PDF Folder
Adjust `PDF_FOLDER` and `IMAGE_DIR` to your paths.

In [ ]:
# =============================================================================
# EXAMPLE E - Process app_data PDF folder only (recursive)
# =============================================================================

from pathlib import Path

PDF_FOLDER = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\app_data"
IMAGE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\images\app_data_images"

# Optional: reset all_docs before processing a fresh folder
# all_docs = []

pdf_root = Path(PDF_FOLDER)
if not pdf_root.exists():
    print(f"[ERROR] app_data folder not found: {pdf_root}")
else:
    pdf_count = len(list(pdf_root.rglob("*.pdf")))
    print(f"[INFO] app_data folder exists: {pdf_root}")
    print(f"[INFO] app_data PDFs discovered : {pdf_count}")

    build_chunk_documents_from_pdf_folder(
        folder_path=PDF_FOLDER,
        image_output_dir=IMAGE_DIR,
        chunk_size=300,
        chunk_overlap=50,
        extract_images=True,
        save_json=True,
    )

    inspect_all_docs(sample_n=3)

[INFO] PDF folder exists: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\rainfall_data\indian gvn reports
[INFO] PDFs discovered : 12
[FOLDER] Found 12 PDF(s) under: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\rainfall_data\indian gvn reports
[FOLDER] Images → C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\images\rainfall_images
------------------------------------------------------------

[1/12] Processing: Basics-of-SWH.pdf
[INFO] 'Basics-of-SWH.pdf' — 14 pages
  → Extracting text page 14/14...
[IMAGES] Extracted 3 images → C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\images\rainfall_images
[SAVED JSON] C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\rainfall_data\indian gvn reports\Basics-of-SWH.json
[all_docs] Total documents after PDF: 15496
  ✓  +166 docs  (text=158, table=5, image=3)  → all_docs total: 15496

[2/12] Processing: Central Water Commission_Daily_Flood_situation_repor

In [ ]:
import os
import pickle

path = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\text_data\app_info"
os.makedirs(path, exist_ok=True)

output_file = os.path.join(path, "app_data_docs.pkl")

# Safety: persist only app_data docs
app_only_docs = [d for d in all_docs if d.metadata.get("dataset") == "app_data"]

with open(output_file, "wb") as f:
    pickle.dump(app_only_docs, f)

print(f"app_data docs saved successfully at: {output_file}")
print(f"Total app_data docs saved: {len(app_only_docs)}")

all_docs saved successfully at: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\text_data\app_info\flood_app_data.pkl
Total docs saved: 125
